In [1]:
from __future__ import print_function
import sys, os, glob, subprocess
import math
import multiprocessing

import ROOT
from ROOT import gROOT, gStyle, gDirectory, gPad
nthreads = min(4, max(1, multiprocessing.cpu_count()-1))  # tune as needed
ROOT.EnableImplicitMT(nthreads)
gStyle.SetOptStat(1111111)

from pdgid import pdgid_dict
import plot_utils
import constants
import log_parser

# location of catalog files
MuBeamStopVD_catalog = '/exp/mu2e/data/users/yongyiwu/nucap/datasets/MDC2020/catalog/nts.yongyiwu.MDC2020s21r0000.nucap.root.files'
MuminusStopToVD_catalog = '/exp/mu2e/data/users/yongyiwu/nucap/datasets/MDC2020/catalog/nts.yongyiwu.MDC2020s22r0000.nucap.root.files'

with open(MuBeamStopVD_catalog, 'r') as f:
    MuBeamStopVD_rootlist = f.read().splitlines()
with open(MuminusStopToVD_catalog, 'r') as f:
    MuminusStopToVD_rootlist = f.read().splitlines()

In [2]:
nStoppedMuminus_MuBeamStopVD = 107501479. # from running log_parser.py
nStoppedMuminus_MuminusStopToVD = 991.*100000.
# weight normalize to 1e8 stopped muons
weight_MuBeamStopVD = 1000.*100000./nStoppedMuminus_MuBeamStopVD
weight_MuminusStopToVD = 1000.*100000./nStoppedMuminus_MuminusStopToVD

In [3]:
# particles: e+, e-, gamma, n, p, D, T
pdgId_list = [-11, 11, 22, 2112, 2212, 1000010020, 1000010030]
id_names   = ["positron","electron","photon","neutron","proton","D","T"]

tmp_dir = "/exp/mu2e/data/users/yongyiwu/stm_analysis/tmp"

run_types = ["MuBeamStopVD", "MuminusStopToVD"]
#root_lists = [MuBeamStopVD_rootlist[:20], MuminusStopToVD_rootlist[:10]]
root_lists = [MuBeamStopVD_rootlist, MuminusStopToVD_rootlist]
weights = [weight_MuBeamStopVD, weight_MuminusStopToVD]

# time wise: muminus lifetime similar enough. propagating small distance, time making little sense
# look at pz, pt, x, y, startz, E (gamma needs 3 zoomed in areas), 
# r-pz, r-pt, x-y

for run_type, filelist, weight in zip(run_types, root_lists, weights):
    print(f"Producing merged hist file for {run_type} from {len(filelist)} inputs using {nthreads} threads")

    merged_path = os.path.join(tmp_dir, f"hist_{run_type}_merged.root")
    out_f = ROOT.TFile(merged_path, "RECREATE")

    # Create a single RDataFrame over the entire file list
    rdf = ROOT.RDataFrame("ParticleBackTracer/ttree", filelist)

    # define derived columns once
    rdf = rdf.Filter("pz > 0.")
    rdf = rdf.Define("pt", "1000.*sqrt(px*px + py*py)") \
             .Redefine("pz", "1000.*pz") \
             .Redefine("E", "1000.*E") \
             .Define("r",  "sqrt((x+3904.0)*(x+3904.0) + y*y)")
    
    # produce histograms (a small fixed number per particle)
    for pdgId, pid in zip(pdgId_list, id_names):
        rdfp = rdf.Filter(f"pdgId == {pdgId}")

        pmax, pnbin, pbinsize = 0., 0., 0.
        if pdgId == -11:
            pmax = 20000.
            pnbin = 500
            pbinsize = pmax/pnbin
            Emax = pmax
            Ebinsize = pbinsize
        elif pdgId == 11:
            pmax = 60000.
            pnbin = 500
            pbinsize = pmax/pnbin
            Emax = pmax
            Ebinsize = pbinsize
        elif pdgId == 22:
            pmax = 8000.
            pnbin = 1000
            pbinsize = pmax/pnbin
            Emax = pmax
            Ebinsize = pbinsize
        elif pdgId == 2112:
            pmax = 50000.
            pnbin = 1000
            pbinsize = pmax/pnbin
            Emax = pmax
            Ebinsize = pbinsize
        else:
            pmax = 200000.
            pnbin = 500
            pbinsize = pmax/pnbin
            Emax = pmax/2.5
            Ebinsize = pbinsize*2.5

        if run_type == "MuBeamStopVD":
            title_prefix = "Stock G4:"
        else:
            title_prefix = "Mu2e mu resampler:"
        
        # 1D histograms (names include particle identifier pid)
        hist_pz = rdfp.Histo1D((f"{run_type}_{pid}_pz", 
                                 f"{title_prefix} {pid} momentum pz;pz [keV];counts per {pbinsize:.0f}keV", 
                                 pnbin, 0., pmax), "pz")
        hist_pt = rdfp.Histo1D((f"{run_type}_{pid}_pt", 
                                 f"{title_prefix} {pid} transverse momentum pT;pt [keV];counts per {pbinsize:.0f}keV", 
                                 pnbin, 0., pmax), "pt")
        hist_x  = rdfp.Histo1D((f"{run_type}_{pid}_x", 
                                 f"{title_prefix} {pid} x position;x [mm];counts per 5mm", 
                                 240, constants.VD101_X-600., constants.VD101_X+600.), "x")
        hist_y  = rdfp.Histo1D((f"{run_type}_{pid}_y", 
                                 f"{title_prefix} {pid} y position;y [mm];counts per 5mm", 
                                 240, -600., 600.), "y")
        hist_startz = rdfp.Histo1D((f"{run_type}_{pid}_startz", 
                                     f"{title_prefix} {pid} start z;startz [mm];counts per 20mm", 
                                     200, 3000., 7000.), "startz")
        hist_E  = rdfp.Histo1D((f"{run_type}_{pid}_E", 
                                 f"{title_prefix} {pid} kin. energy;E [keV];counts per {Ebinsize:.0f}keV", 
                                 pnbin, 0., Emax), "E")
        
        extra_hist_E = []
        if pid == "photon":
            hist_E_zoom1 = rdfp.Histo1D((f"{run_type}_{pid}_E_zoom1", f"{title_prefix} photon E (325-355 keV);E [keV];counts per 1keV", 30, 325., 355.), "E")
            hist_E_zoom2 = rdfp.Histo1D((f"{run_type}_{pid}_E_zoom2", f"{title_prefix} photon E (830-860 keV);E [keV];counts per 1keV", 30, 830., 860.), "E")
            hist_E_zoom3 = rdfp.Histo1D((f"{run_type}_{pid}_E_zoom3", f"{title_prefix} photon E (1795-1825 keV);E [keV];counts per 1keV", 30, 1795., 1825.), "E")
            extra_hist_E.extend([hist_E_zoom1, hist_E_zoom2, hist_E_zoom3])
        if pid == "neutron":
            hist_pz_zoom = rdfp.Histo1D((f"{run_type}_{pid}_pz_zoom", 
                                         f"{title_prefix} {pid} momentum pz zoomed;pz [keV];counts per 10keV", 
                                         200, 0., 2000.), "pz")
            hist_pt_zoom = rdfp.Histo1D((f"{run_type}_{pid}_pt_zoom", 
                                         f"{title_prefix} {pid} transverse momentum pT zoomed;pt [keV];counts per 10keV", 
                                         200, 0., 2000.), "pt")
            hist_E_zoom  = rdfp.Histo1D((f"{run_type}_{pid}_E_zoom", 
                                         f"{title_prefix} {pid} kin. energy zoomed;E [keV];counts per 10keV", 
                                         200, 0., 2000.), "E")
            extra_hist_E.extend([hist_pz_zoom, hist_pt_zoom, hist_E_zoom])

        # 2D histograms
        hist2_r_pz  = rdfp.Histo2D((f"{run_type}_{pid}_pz_vs_r", f"{title_prefix} {pid} pz vs r;r per 5mm;pz per {5*pbinsize:.0f}keV", 120, 0., 600., pnbin//5, 0., pmax), "r", "pz")
        hist2_r_pt  = rdfp.Histo2D((f"{run_type}_{pid}_pt_vs_r", f"{title_prefix} {pid} pt vs r;r per 5mm;pt per {5*pbinsize:.0f}keV", 120, 0., 600., pnbin//5, 0., pmax), "r", "pt")
        hist2_pz_pt = rdfp.Histo2D((f"{run_type}_{pid}_pt_vs_pz", f"{title_prefix} {pid} pt vs pz;pz per {5*pbinsize:.0f}keV;pt per {5*pbinsize:.0f}keV", pnbin//5, 0., pmax, pnbin//5, 0., pmax), "pz", "pt")
        hist2_x_y   = rdfp.Histo2D((f"{run_type}_{pid}_y_vs_x", f"{title_prefix} {pid} y vs x;x per 10mm;y per 10mm", 120, constants.VD101_X-600., constants.VD101_X+600., 120, -600., 600.), "x", "y")

        # Collect all RResultPtr objects to force evaluation and write to file
        all_results = [hist_pz, hist_pt, hist_x, hist_y, hist_startz, hist_E,
                       hist2_r_pz, hist2_r_pt, hist2_pz_pt, hist2_x_y] + extra_hist_E

        # GetValue() triggers the RDataFrame action and returns the actual histogram object (TH1/TH2)
        # Write each histogram into the output file
        for rptr in all_results:
            h = rptr.GetValue()
            h.Scale(float(weight))
            h.Write()

    out_f.Close()
    print(f"Wrote merged file: {merged_path}")

Producing merged hist file for MuBeamStopVD from 2000 inputs using 3 threads
Wrote merged file: /exp/mu2e/data/users/yongyiwu/stm_analysis/tmp/hist_MuBeamStopVD_merged.root
Producing merged hist file for MuminusStopToVD from 991 inputs using 3 threads
Wrote merged file: /exp/mu2e/data/users/yongyiwu/stm_analysis/tmp/hist_MuminusStopToVD_merged.root
